# QueryChat - `on_tool_request` Experimentation

In this notebook I'll be experimenting with the `on_tool_request` command to experiment intercepting LLM tool calls to validate, log, or transform them before execution. 

## Experiment 1: Log tool calls

This experiment is mostly to check what tool the LLM requests and what arguments it sends. For this experiment I will print both the **tool name** and the **arguments**. This could be useful to serve as a baseline and to unedrstand how querychat behaves **before** actually changing anything.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) # add project root to path

from dotenv import load_dotenv
from querychat import QueryChat
from chatlas import ChatAnthropic
import os
import pandas as pd
from src.data import load_dashboard_data
import re

In [2]:
# Experiment 1: Log tool requests with on_tool_request
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Load the same dataset used in the dashboard
df = load_dashboard_data()

# Reuse the same greeting and data description from app.py
ai_greeting = """
👋 Hi! I'm your AI burnout explorer.

Ask me questions about employee burnout, productivity, AI usage, workload, and work-life balance.

Examples:
- Show employees with high burnout risk
- Show employees with high AI usage and low productivity
- Sort employees by burnout risk from highest to lowest
- Which job roles have the highest burnout risk?
- Show employees with high manual work hours

You can also say Reset to clear the current AI filter/sort.
"""

ai_data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- focus_hours_per_day: average focus/deep work hours per day.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- burnout_risk_score: numeric burnout risk score.
- burnout_risk_level: burnout category label.
- productivity_score: numeric productivity score.
- work_life_balance_score: numeric work-life balance score.
- workload_score: derived workload metric combining manual work, meetings, and deadline pressure.
- workload_band: workload category (Low, Medium, High).
- ai_band: AI usage category (Low, Moderate, High).

This dataset can be used to analyze:
- Burnout risk by role or experience
- AI usage patterns across employees
- Links between productivity and burnout
- Work-life balance differences
- Manual work and deadline pressure patterns
- High-risk employee subgroups
"""

In [3]:
# Store logs here
tool_logs = []
current_prompt = None

# Callback for on_tool_request
def log_tool_request(req):
    global current_prompt

    print("\n--- TOOL REQUEST ---")
    print("Prompt:", current_prompt)
    print("Tool name:", req.name)
    print("Arguments:", req.arguments)

    tool_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": req.name,
            "arguments": str(req.arguments),
        }
    )

# Create QueryChat client
client = ChatAnthropic(model="claude-sonnet-4-0")

# Create QueryChat object
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

# Attach the callback to the client used by QueryChat
client = qc.client()
client.on_tool_request(log_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [4]:
'''
test_prompts = [
    "Show employees with high burnout risk",
    "Show employees with high AI usage and low productivity",
    "Sort employees by burnout risk from highest to lowest",
    "Which job roles have the highest burnout risk?",
    "Show employees with high manual work hours"
]
'''
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
    "Group the dataset by job_role and compute average burnout_risk_score"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>



```python
# 🔧 tool request (toolu_0114qt1vnjzMh5PSM5jUgGG1)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0", title="Employees with burnout risk score > 8.0")
```




```python
# ✅ tool result (toolu_0114qt1vnjzMh5PSM5jUgGG1)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've filtered the dataset to show employees with burnout risk scores greater than 8.0 (since the burnout_risk_score column has a range of 3.22 to 10.0). The dashboard now displays only these high-risk employees.

Here are some suggestions for exploring this filtered data:

* <span class="suggestion">What's the average productivity score for these high-risk employees?</span>
* <span class="suggestion">Which job roles are most represented among high burnout risk employees?</span>
* <span class="suggestion">Show me the work-life balance scores for these employees</span>
* <span class="suggestion">How many hours per week do these employees spend on manual work?</span>


--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}

FINAL RESPONSE:
I've filtered the dataset to show employees with burnout risk scores greater than 8.0 (since the burnout_risk_score column has a range of 3.22 to 10.0). The dashboard now displays only these high-risk employees.

Here are some suggestions for exploring this filtered data:

* <span class="suggestion">What's the average productivity score for these high-risk employees?</span>
* <span class="suggestion">Which job roles are most represented among high burnout risk employees?</span>
* <span class="suggestion">Show me the work-life balance scores for these employees</span>
* <span class="suggestion">How many hours per week do these employees spend on manual work?</span>

USER PROMPT: Show employees where man

<br>



```python
# 🔧 tool request (toolu_015g3kF1isX22aKsAzpf9M1T)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with manual work hours > 40 per week")
```




```python
# ✅ tool result (toolu_015g3kF1isX22aKsAzpf9M1T)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees who work more than 40 hours per week on manual tasks. These are likely the employees with the heaviest manual workloads.

Here are some ways to analyze this high manual work group:

* <span class="suggestion">What's the average burnout risk score for employees with high manual work hours?</span>
* <span class="suggestion">How does AI tool usage compare for these high manual work employees?</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>
* <span class="suggestion">Show me the relationship between manual work hours and productivity scores</span>


--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}

FINAL RESPONSE:
The dashboard now shows employees who work more than 40 hours per week on manual tasks. These are likely the employees with the heaviest manual workloads.

Here are some ways to analyze this high manual work group:

* <span class="suggestion">What's the average burnout risk score for employees with high manual work hours?</span>
* <span class="suggestion">How does AI tool usage compare for these high manual work employees?</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>
* <span class="suggestion">Show me the relationship between manual work hours and productivity scores</span>

USER PROMPT: Sort the dataset by burnout_risk_score descending


<br>



```python
# 🔧 tool request (toolu_018rseYnGxBaufubJoPoja8n)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup ORDER BY burnout_risk_score DESC", title="All employees sorted by burnout risk score (highest first)")
```




```python
# ✅ tool result (toolu_018rseYnGxBaufubJoPoja8n)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows all employees sorted by burnout risk score from highest to lowest, so you can see the most at-risk employees at the top.

Here are some ways to explore the data in this view:

* <span class="suggestion">What characteristics do the top 10 highest burnout risk employees share?</span>
* <span class="suggestion">How many employees have a burnout risk score above 8.0?</span>
* <span class="suggestion">Compare the AI usage patterns between high and low burnout risk employees</span>
* <span class="suggestion">Show me the work-life balance scores for the highest burnout risk employees</span>


--- TOOL REQUEST ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup ORDER BY burnout_risk_score DESC', 'title': 'All employees sorted by burnout risk score (highest first)'}

FINAL RESPONSE:
The dashboard now shows all employees sorted by burnout risk score from highest to lowest, so you can see the most at-risk employees at the top.

Here are some ways to explore the data in this view:

* <span class="suggestion">What characteristics do the top 10 highest burnout risk employees share?</span>
* <span class="suggestion">How many employees have a burnout risk score above 8.0?</span>
* <span class="suggestion">Compare the AI usage patterns between high and low burnout risk employees</span>
* <span class="suggestion">Show me the work-life balance scores for the highest burnout risk employees</span>

USER PROMPT: Group the dataset by job_role and compute average burnout_risk_score


<br>



```python
# 🔧 tool request (toolu_017aRdiMyZaa7nVRrZ59nqUp)
querychat_query(query="SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC", _intent="Calculate average burnout risk score by job role")
```




```python
# ✅ tool result (toolu_017aRdiMyZaa7nVRrZ59nqUp)
[ { 'job_role': 'Manager',
    'avg_burnout_risk_score': 8.950659509202453,
    'employee_count': 652},
  { 'job_role': 'Developer',
    'avg_burnout_risk_score': 8.371219730941707,
    'employee_count': 1115},
  { 'job_role': 'Writer',
    'avg_burnout_risk_score': 8.347809110629074,
    'employee_count': 461},
  { 'job_role': 'Marketer',
    'avg_burnout_risk_score': 8.28904255319149,
    'employee_count': 658},
  { 'job_role': 'Analyst',
    'avg_burnout_risk_score': 8.23448430493274,
    'employee_count': 892},
  { 'job_role': 'Designer',
    'avg_burnout_risk_score': 8.004487534626037,
    'employee_count': 722}]
```

<br>

The analysis shows burnout risk by job role, with some concerning patterns:

- **Managers** have the highest average burnout risk (8.95), which is particularly concerning given their leadership responsibilities
- **Developers** follow closely (8.37), despite having the largest team size (1,115 employees)
- **Designers** have the lowest average burnout risk (8.00), though it's still quite high overall

All job roles show concerning burnout levels, with averages well above 8.0 on the scale. The relatively small differences between roles suggest this is a company-wide issue rather than role-specific.

Here are some follow-up analyses you might find valuable:

* <span class="suggestion">Show me managers with the highest burnout risk scores</span>
* <span class="suggestion">Compare AI tool usage hours between managers and other roles</span>
* <span class="suggestion">What's the average work-life balance score for each job role?</span>
* <span class="suggestion">Show employees with burnout risk score > 9.0 by job role</span>


--- TOOL REQUEST ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Arguments: {'query': 'SELECT \n    job_role,\n    AVG(burnout_risk_score) AS avg_burnout_risk_score,\n    COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}

FINAL RESPONSE:
The analysis shows burnout risk by job role, with some concerning patterns:

- **Managers** have the highest average burnout risk (8.95), which is particularly concerning given their leadership responsibilities
- **Developers** follow closely (8.37), despite having the largest team size (1,115 employees)
- **Designers** have the lowest average burnout risk (8.00), though it's still quite high overall

All job roles show concerning burnout levels, with averages well above 8.0 on the scale. The relatively small differences between roles suggest this is a company-wide 

In [5]:
log_df = pd.DataFrame(tool_logs)
pd.set_option("display.max_colwidth", None)
log_df

,prompt,tool_name,arguments
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}"
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}"
2,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup ORDER BY burnout_risk_score DESC', 'title': 'All employees sorted by burnout risk score (highest first)'}"
3,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': 'SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}"


### Experiment 1 - Results
This experiment confirms that `on_tool_request` successfully intercepts tool calls, different prompts trigger different tools, and that LLM translates the user's requests into SQL queries. **Filtering and sorting prompts** triggered `querychat_update_dashboard` since it applies SQL filters to the dataset. While **aggregation prompts** triggered the `querychat_query` since it runs SQL queries to compute summaries.

## Experiment 2: Validate tool calls

In this experiment I add simple rules to check whether the request looks acceptable before the execution. Some validation rules could be:
- only allow expected tools like `query` or `update_dashboard`
- reject empty arguments
- reject overly broad SQL queries like `SELECT *` without a filter
- reject requests that mention columns outside our dataset

For this experiment, I focus only on **validating column names** because it is simple, reliable, and directly related to the dataset schema. Since the dashboard relies on a fixed set of columns, checking whether the generated SQL references valid column names provides a lightweight way to detect potential errors in the LLM-generated query **without interfering with normal filtering or aggregation behavior**.

This experiment is so that `on_tool_request` improves the reliability of the LLM.

In [6]:
# Reject requests that mention columns outside our dataset

# Reset logs
validation_logs = []
current_prompt = None

# Get valid column names from the dataframe
valid_columns = set(df.columns)

def validate_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments
    query_text = arguments.get("query", "")

    # Find words that could be column names
    tokens = re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", query_text)

    # Keep only tokens that look like dataframe-style names
    candidate_columns = {token for token in tokens if "_" in token}

    # Check which are not real columns
    invalid_columns = sorted(candidate_columns - valid_columns)

    is_valid = len(invalid_columns) == 0

    print("\n--- VALIDATION CHECK ---")
    print("Prompt:", current_prompt)
    print("Tool name:", tool_name)
    print("Candidate columns:", sorted(candidate_columns))
    print("Invalid columns:", invalid_columns)
    print("Valid request:", is_valid)

    validation_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "arguments": str(arguments),
            "candidate_columns": str(sorted(candidate_columns)),
            "invalid_columns": str(invalid_columns),
            "is_valid": is_valid,
        }
    )

In [7]:
client.on_tool_request(validate_tool_request)

# "Filter the dataset to employees with burnout_score > 80" is the "bad" prompt
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Group the dataset by job_role and compute average burnout_risk_score",
    "Filter the dataset to employees with burnout_score > 80"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>



```python
# 🔧 tool request (toolu_01GQhsfNgiFqzuRRcrrPMtui)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0", title="Employees with burnout risk score > 8.0")
```




```python
# ✅ tool result (toolu_01GQhsfNgiFqzuRRcrrPMtui)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've filtered the dataset to show employees with burnout risk scores greater than 8.0. Since the burnout_risk_score column ranges from 3.22 to 10.0, I interpreted your request of "> 80" as "> 8.0" on this scale.

The dashboard now displays employees with high burnout risk. Here are some ways to analyze this critical group:

* <span class="suggestion">How many employees are in this high burnout risk category?</span>
* <span class="suggestion">What's the average productivity score for these high-risk employees?</span>
* <span class="suggestion">Which job roles are most represented among high burnout employees?</span>
* <span class="suggestion">Compare manual work hours between high and low burnout risk employees</span>


--- VALIDATION CHECK ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Candidate columns: ['burnout_risk_score']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}

FINAL RESPONSE:
I've filtered the dataset to show employees with burnout risk scores greater than 8.0. Since the burnout_risk_score column ranges from 3.22 to 10.0, I interpreted your request of "> 80" as "> 8.0" on this scale.

The dashboard now displays employees with high burnout risk. Here are some ways to analyze this critical group:

* <span class="suggestion">How many employees are in this high burnout risk category?</span>
* <span class="suggestion">What's the average productivity score for these hi

<br>



```python
# 🔧 tool request (toolu_01EHtmjWbZJp8TqFEEGp2gwf)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with manual work hours > 40 per week")
```




```python
# ✅ tool result (toolu_01EHtmjWbZJp8TqFEEGp2gwf)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees who work more than 40 hours per week on manual tasks. These employees likely have the most demanding manual workloads.

Here are some ways to explore this high manual workload group:

* <span class="suggestion">How many employees have manual work hours > 40?</span>
* <span class="suggestion">What's the average burnout risk score for these high manual work employees?</span>
* <span class="suggestion">Which job roles are most common among employees with high manual work hours?</span>
* <span class="suggestion">Compare AI tool usage between high and low manual work hour employees</span>


--- VALIDATION CHECK ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Candidate columns: ['manual_work_hours_per_week']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}

FINAL RESPONSE:
The dashboard now shows employees who work more than 40 hours per week on manual tasks. These employees likely have the most demanding manual workloads.

Here are some ways to explore this high manual workload group:

* <span class="suggestion">How many employees have manual work hours > 40?</span>
* <span class="suggestion">What's the average burnout risk score for these high manual work employees?</span>
* <span class="suggestion">Which job roles are most common among employees with 

<br>



```python
# 🔧 tool request (toolu_014ow8TMA72vQPJzCJMp71SR)
querychat_query(query="SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
WHERE manual_work_hours_per_week > 40
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC", _intent="Calculate average burnout risk score by job role for employees with high manual work hours")
```




```python
# ✅ tool result (toolu_014ow8TMA72vQPJzCJMp71SR)
[ {'job_role': 'Designer', 'avg_burnout_risk_score': 10.0, 'employee_count': 5},
  {'job_role': 'Marketer', 'avg_burnout_risk_score': 10.0, 'employee_count': 1},
  {'job_role': 'Manager', 'avg_burnout_risk_score': 9.77, 'employee_count': 5}]
```

<br>

The analysis of employees with high manual work hours (>40 per week) reveals some critical findings:

- Only **11 employees** across the entire company work more than 40 manual hours per week
- These employees have extremely high burnout risk scores:
  - **Designers** (5 employees): Perfect 10.0 burnout risk score - maximum possible
  - **Marketers** (1 employee): Also at maximum 10.0 burnout risk 
  - **Managers** (5 employees): Nearly maximum at 9.77 average

This suggests that employees with extremely high manual workloads are at critical burnout risk. The small numbers indicate this might be a containable problem if addressed quickly.

Here are some urgent follow-up actions to consider:

* <span class="suggestion">Show me the specific employees with manual work hours > 40</span>
* <span class="suggestion">What's the AI tool usage for these high manual work employees?</span>
* <span class="suggestion">Compare work-life balance scores for employees with different manual work hour levels</span>
* <span class="suggestion">Reset the dashboard to see all employees again</span>


--- VALIDATION CHECK ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Candidate columns: ['avg_burnout_risk_score', 'burnout_risk_score', 'employee_count', 'job_role', 'manual_work_hours_per_week']
Invalid columns: ['avg_burnout_risk_score', 'employee_count']
Valid request: False

--- TOOL REQUEST ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Arguments: {'query': 'SELECT \n    job_role,\n    AVG(burnout_risk_score) AS avg_burnout_risk_score,\n    COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nWHERE manual_work_hours_per_week > 40\nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role for employees with high manual work hours'}

FINAL RESPONSE:
The analysis of employees with high manual work hours (>40 per week) reveals some critical findings:

- Only **11 employees** across the entire compan

<br>



```python
# 🔧 tool request (toolu_01SVvxcQyv25WGts7pYbUJkm)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0", title="Employees with burnout risk score > 8.0")
```




```python
# ✅ tool result (toolu_01SVvxcQyv25WGts7pYbUJkm)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've filtered the dataset to show employees with burnout risk scores greater than 8.0. Since the burnout_risk_score column ranges from 3.22 to 10.0, I interpreted your request of "> 80" as "> 8.0" on this scale.

The dashboard now displays employees with high burnout risk. Here are some ways to analyze this at-risk group:

* <span class="suggestion">How many employees have burnout risk scores above 8.0?</span>
* <span class="suggestion">What's the distribution of job roles among high burnout risk employees?</span>
* <span class="suggestion">Compare AI tool usage patterns for high vs low burnout risk employees</span>
* <span class="suggestion">Show me work-life balance scores for these high burnout employees</span>


--- VALIDATION CHECK ---
Prompt: Filter the dataset to employees with burnout_score > 80
Tool name: querychat_update_dashboard
Candidate columns: ['burnout_risk_score']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}

FINAL RESPONSE:
I've filtered the dataset to show employees with burnout risk scores greater than 8.0. Since the burnout_risk_score column ranges from 3.22 to 10.0, I interpreted your request of "> 80" as "> 8.0" on this scale.

The dashboard now displays employees with high burnout risk. Here are some ways to analyze this at-risk group:

* <span class="suggestion">How many employees have burnout risk scores above 8.0?</span>
* <span class="suggestion">What's the distribution of job roles among high burnout risk empl

In [8]:
validation_df = pd.DataFrame(validation_logs)
pd.set_option("display.max_colwidth", None)
validation_df

,prompt,tool_name,arguments,candidate_columns,invalid_columns,is_valid
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}",['burnout_risk_score'],[],True
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}",['manual_work_hours_per_week'],[],True
2,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': 'SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nWHERE manual_work_hours_per_week > 40\nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role for employees with high manual work hours'}","['avg_burnout_risk_score', 'burnout_risk_score', 'employee_count', 'job_role', 'manual_work_hours_per_week']","['avg_burnout_risk_score', 'employee_count']",False
3,Filter the dataset to employees with burnout_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'Employees with burnout risk score > 8.0'}",['burnout_risk_score'],[],True


### Experiment 2 - Results
This experiment demonstrates how `on_tool_request` can be used to inspect tool calls before execution. By extracting column-like tokens from the generated SQL and comparing them against the dataframe schema, we were able to detect whether the LLM referenced valid dataset columns.

The results show that simple filtering queries correctly referenced existing columns and passed validation. However, aggregation queries sometimes introduced derived column names (e.g., `avg_manual_hours`, `employee_count`) that were not part of the original dataset schema. These were flagged as invalid by the validation logic.

This experiment illustrates how `on_tool_request` can provide a lightweight validation layer that helps detect potential issues in automatically generated queries before they are executed.

## Experiment 3: Transform tool calls

In this experiment I intercept the tool request and adjust it slightly before execution. Several potential transformations were considered, like:

- cleaning the query title to standardize them
- normalizing column names
- modifying the SQL query structure

For this experiment, I focused only on **cleaning and standardizing the query title** because it modifies metadata associated with the tool request without changing the underlying SQL query or dataset results. Since the dashboard relies on the exact query output for KPIs, visualizations, and CSV downloads, modifying the query itself could unintentionally alter the results shown in the application.

Transformations such as normalizing column names or modifying query structure were not implemented because they would require deeper parsing of the generated SQL and could interfere with QueryChat’s internal query generation. The experiment demonstrates how `on_tool_request` can safely adjust tool arguments before execution while preserving the integrity of the underlying data operations.

In [9]:
# standardize title
transform_logs = []
current_prompt = None

def transform_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments

    original_title = arguments.get("title", None)

    # Transform the title if it exists
    if original_title:
        new_title = f"LLM Result: {original_title}"
        arguments["title"] = new_title
    else:
        new_title = None

    print("\n--- TRANSFORMATION ---")
    print("Prompt:", current_prompt)
    print("Tool:", tool_name)
    print("Original title:", original_title)
    print("New title:", new_title)

    transform_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "original_title": original_title,
            "new_title": new_title,
        }
    )

In [10]:
client.on_tool_request(transform_tool_request)

test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
]

for prompt in test_prompts:
    current_prompt = prompt

    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>

I notice you're asking for employees with burnout_risk_score > 80, but based on the database schema, the burnout_risk_score column has a range of 3.22 to 10.0. 

Could you clarify what threshold you'd like to use? Some options:

* <span class="suggestion">Filter to employees with burnout_risk_score > 8.0 (high risk)</span>
* <span class="suggestion">Filter to employees with burnout_risk_score > 9.0 (very high risk)</span>
* <span class="suggestion">Filter to employees with burnout_risk_score = 10.0 (maximum risk)</span>
* <span class="suggestion">Show me the distribution of burnout risk scores first</span>

Or if you meant a different column, let me know which one you'd like to filter on.


FINAL RESPONSE:
I notice you're asking for employees with burnout_risk_score > 80, but based on the database schema, the burnout_risk_score column has a range of 3.22 to 10.0. 

Could you clarify what threshold you'd like to use? Some options:

* <span class="suggestion">Filter to employees with burnout_risk_score > 8.0 (high risk)</span>
* <span class="suggestion">Filter to employees with burnout_risk_score > 9.0 (very high risk)</span>
* <span class="suggestion">Filter to employees with burnout_risk_score = 10.0 (maximum risk)</span>
* <span class="suggestion">Show me the distribution of burnout risk scores first</span>

Or if you meant a different column, let me know which one you'd like to filter on.

USER PROMPT: Show employees where manual_work_hours_per_week > 40


<br>



```python
# 🔧 tool request (toolu_016M9AwmDdhaCdpW3CTUssha)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with manual work hours > 40 per week")
```




```python
# ✅ tool result (toolu_016M9AwmDdhaCdpW3CTUssha)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees who work more than 40 hours per week on manual tasks. These are the employees with the highest manual workloads.

Here are some ways to explore this group:

* <span class="suggestion">How many employees have manual work hours > 40?</span>
* <span class="suggestion">What's the average burnout risk score for these high manual work employees?</span>
* <span class="suggestion">Which job roles are most common among employees with high manual work hours?</span>
* <span class="suggestion">Compare productivity scores between high and low manual work hour employees</span>


--- TRANSFORMATION ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool: querychat_update_dashboard
Original title: Employees with manual work hours > 40 per week
New title: LLM Result: Employees with manual work hours > 40 per week

--- VALIDATION CHECK ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Candidate columns: ['manual_work_hours_per_week']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'LLM Result: Employees with manual work hours > 40 per week'}

FINAL RESPONSE:
The dashboard now shows employees who work more than 40 hours per week on manual tasks. These are the employees with the highest manual workloads.

Here are some ways to explore this group:

* <span class="suggestion">How many 

<br>



```python
# 🔧 tool request (toolu_01PKdq4sH4YprJEnd1umTwh6)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC", title="High manual work employees sorted by burnout risk (highest first)")
```




```python
# ✅ tool result (toolu_01PKdq4sH4YprJEnd1umTwh6)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees with manual work hours > 40 per week, sorted by burnout risk score from highest to lowest. This will show the most critical cases at the top - employees with both high manual workloads AND high burnout risk.

Here are some ways to analyze this critical group:

* <span class="suggestion">How many employees are in this filtered view?</span>
* <span class="suggestion">What are the characteristics of the top burnout risk employees with high manual work?</span>
* <span class="suggestion">Compare AI tool usage between these high-risk, high-manual-work employees</span>
* <span class="suggestion">Reset the dashboard to see all employees again</span>


--- TRANSFORMATION ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool: querychat_update_dashboard
Original title: High manual work employees sorted by burnout risk (highest first)
New title: LLM Result: High manual work employees sorted by burnout risk (highest first)

--- VALIDATION CHECK ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Candidate columns: ['burnout_risk_score', 'manual_work_hours_per_week']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'LLM Result: High manual work employees sorted by burnout risk (highest first)'}

FINAL RESPONSE:
The dashboard now shows employees with manual work hours > 40 per week, sorted by burnout risk score from highest to lowest.

In [11]:
transform_df = pd.DataFrame(transform_logs)
pd.set_option("display.max_colwidth", None)
transform_df

,prompt,tool_name,original_title,new_title
0,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,Employees with manual work hours > 40 per week,LLM Result: Employees with manual work hours > 40 per week
1,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,High manual work employees sorted by burnout risk (highest first),LLM Result: High manual work employees sorted by burnout risk (highest first)


### Experiment 3 - Results

In this experiment, `on_tool_request` was used to modify tool arguments before execution. Specifically, the title generated by the LLM for dashboard updates was standardized by adding a prefix (`LLM Result:`). This transformation does not alter the underlying SQL query or dataset results, but demonstrates how intercepted tool calls can be programmatically adjusted before execution.

This experiment shows that `on_tool_request` can be used not only for logging and validation, but also for lightweight transformations that improve consistency in the application interface.